![Noteable.ac.uk Banner](https://raw.githubusercontent.com/YusufNik/Noteable/refs/heads/main/images/Noteable%20NB%20Header%20Banner.png)

## Exemplar Information

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Purpose:</b> This exemplar introduces students to real-world solar physics and space-weather analysis using GOES X-ray flux data. It demonstrates how to load and tidy time-series data, smooth noisy measurements, detect flare peaks, estimate simple event properties, compare quiet and active periods, and explore the results interactively with Plotly and ipywidgets. It also provides a gentle introduction to scientific workflow habits such as reproducible loading functions, clear classification rules, and responsible interpretation of observational data.
</div>

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Intended audience / teaching context:</b> Designed for students in introductory-to-intermediate Python, astronomy, or data analysis teaching sessions. Suitable for classroom demonstration, guided lab work, or independent practice in heliophysics, space-weather, or time-series visualisation topics. Best used by learners who already know basic Python syntax and are ready to work with real scientific data and interactive plots.
</div>

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Noteable Requirements:</b><br>
    <b>Environment:</b> LIVE 2026<br>
    <b>Server:</b> Astronomy<br>
    <b>Kernel:</b> Python 3
</div>

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Required data or dependencies:</b>
    <br>- Internet access for downloading real GOES X-ray time-series data via SunPy/Fido
    - Cached local downloads stored in <code>solar_xrs_cache</code>
    - Python packages: <code>numpy</code>, <code>pandas</code>, <code>matplotlib</code>, <code>plotly</code>, <code>ipywidgets</code>, <code>goes2go</code>, <code>sunpy</code>, <code>IPython</code><br>
    - Plotly submodules: <code>plotly.graph_objects</code>, <code>plotly.io</code>, <code>plotly.subplots</code><br>
    - Standard library modules: <code>pathlib</code>, <code>warnings</code>, <code>logging</code>
</div>

<div style="border:1px solid #ffeeba; background:#fff3cd; color:#856404; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Date created / last reviewed:</b> 13 July 2026<br>
    <b>Maintainer / owner:</b> Nik Yusuf
</div>

# Watching the Active Sun: Interactive Solar Flare and Space-Weather Analysis

## Legend

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    In <b>blue</b>, the <b>instructions</b> and <b>goals</b> are highlighted. This tells you what we are trying to achieve.
</div>
<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    In <b>green</b>, key <b>information</b> and <b>concept explanations</b> are highlighted.
</div>
<div style="border:1px solid #ffeeba; background:#fff3cd; color:#856404; padding:16px; margin:8px 0; border-radius:4px;">
    In <b>yellow</b>, <b>exercises</b> and <b>tasks</b> are highlighted for you to try yourself.
</div>
<div style="border:1px solid #f5c6cb; background:#f8d7da; color:#721c24; padding:16px; margin:8px 0; border-radius:4px;">
    In <b>red</b>, <b>error interpretation</b> and <b>debugging tips</b> are highlighted.
</div>

## 1. Setting Up Our Toolkit

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Goal:</b> Build a beginner-friendly solar-flare analysis workflow using real GOES X-ray flux data. We will inspect an active solar period, identify flare peaks, estimate simple event properties, compare flare windows, and explore the results interactively.
</div>

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Why this matters:</b> Solar flares are not just beautiful bursts of activity. They are dynamic astrophysical events that affect the near-Earth space environment, radio communication, and operational space-weather awareness.
</div>

In this notebook, the main data pathway uses <b>SunPy</b> for GOES X-ray time series, while <b>goes2go</b> is included as part of the Noteable solar toolkit for broader GOES-based exploration.

Click the code cell below and press the **<code>▶</code> Run** button in the toolbar at the top (or use `Shift + Enter`).

In [ ]:
print('Starting setup: importing solar, plotting, and widget libraries...')

from pathlib import Path
import warnings
import logging

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
import ipywidgets as widgets

import goes2go
import sunpy
from sunpy.net import Fido, attrs as a
from sunpy.timeseries import TimeSeries

from IPython.display import display

import sys

def silence_asyncio_del_error(exception, value, traceback, tb=None):
    # Check if this is the specific asyncio cleanup error
    if "BaseEventLoop.__del__" in str(exception) or "ValueError: signal only works in main thread" in str(value):
        return  # Ignore it entirely
    # Fallback to default behavior for all other actual errors
    sys.__excepthook__(exception, value, traceback)

# Apply the hook to unhandled exceptions in hooks/threads
sys.excepthook = silence_asyncio_del_error

# For Python 3.8+, also catch it if it triggers inside a threading context
import threading
if hasattr(threading, 'excepthook'):
    def silence_thread_error(args):
        if "BaseEventLoop.__del__" in str(args.exc_value) or "signal only works" in str(args.exc_value):
            return
        sys.__excepthook__(args.exc_type, args.exc_value, args.exc_traceback)
    threading.excepthook = silence_thread_error

warnings.filterwarnings('ignore')
plt.style.use('default')
pio.renderers.default = 'notebook_connected'

cache_dir = Path('solar_xrs_cache')
cache_dir.mkdir(exist_ok=True)

print(f"SunPy version: {getattr(sunpy, '__version__', 'unknown')}")
print(f"goes2go version: {getattr(goes2go, '__version__', 'unknown')}")
print('Success! Imports are ready. We can now inspect real solar flare timelines.')

## 2. Choosing Event Windows and Understanding the Measurement

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Goal:</b> Work with a small, curated set of solar-activity windows that are short enough for teaching, but rich enough to show real flare behaviour.
</div>

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    <b>What GOES measures:</b> The GOES X-ray Sensor records full-disk solar X-ray flux. In simple language, it tracks how much X-ray power from the whole Sun is reaching the detector. A flare appears as a sudden brightening in this time series.
</div>

A quick rule of thumb for the long-channel X-ray flux:
- <b>C-class</b> begins at $ 10^{-6} $
- <b>M-class</b> begins at $ 10^{-5} $
- <b>X-class</b> begins at $ 10^{-4} $

We will focus on several windows from the very active solar period in May 2024, plus a quieter comparison window.

In [ ]:
print('Creating a curated list of solar activity windows...')

flare_windows = {
    'Quiet comparison window': {
        'start': '2024-05-06 00:00',
        'end': '2024-05-06 06:00',
        'purpose': 'A calmer baseline before the most dramatic flare windows below.'
    },
    'Active-region flare window A': {
        'start': '2024-05-08 04:00',
        'end': '2024-05-08 08:00',
        'purpose': 'An early strong-flare window during the active May 2024 period.'
    },
    'Active-region flare window B': {
        'start': '2024-05-10 12:00',
        'end': '2024-05-10 18:00',
        'purpose': 'A very active interval with strong flare behaviour.'
    },
    'Active-region flare window C': {
        'start': '2024-05-14 14:00',
        'end': '2024-05-14 20:00',
        'purpose': 'A major flare window late in the same active period.'
    }
}

flare_window_df = pd.DataFrame([
    {
        'Event window': label,
        'Start (UTC)': config['start'],
        'End (UTC)': config['end'],
        'Why we chose it': config['purpose']
    }
    for label, config in flare_windows.items()
])

display(flare_window_df)
print('These short windows are designed to download quickly and support clean event comparisons.')

## 3. Loading Real GOES X-ray Data

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Goal:</b> Retrieve real GOES XRS measurements for each curated window, convert them into tidy tables, and calculate a first set of event diagnostics.
</div>

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Good scientific habit:</b> We will use a reproducible loading function and a simple, clearly labelled flare-detection heuristic. That makes it easier to explain what we measured and what the limitations are.
</div>

Run the next cell once. It creates the helper functions we will use throughout the rest of the notebook.

In [ ]:
print('Defining helper functions for loading GOES X-ray data and measuring flare properties...')

def pick_flux_columns(df):
    renamed = {col: str(col).lower() for col in df.columns}
    df = df.rename(columns=renamed).copy()

    short_candidates = [col for col in df.columns if 'xrsa' in col]
    long_candidates = [col for col in df.columns if 'xrsb' in col]

    if len(short_candidates) == 0 or len(long_candidates) == 0:
        raise ValueError(f'Could not identify GOES XRS columns. Available columns: {list(df.columns)}')

    short_col = short_candidates[0]
    long_col = long_candidates[0]
    return df, short_col, long_col


def search_goes_xrs(start_time, end_time, preferred_satellite=18):
    goes_attr_group = getattr(a, 'goes', None)
    satellite_attr = getattr(goes_attr_group, 'SatelliteNumber', None) if goes_attr_group is not None else None

    if satellite_attr is not None:
        query = Fido.search(
            a.Time(start_time, end_time),
            a.Instrument('XRS'),
            satellite_attr(preferred_satellite)
        )
    else:
        query = Fido.search(
            a.Time(start_time, end_time),
            a.Instrument('XRS')
        )

    return query


def load_goes_window(start_time, end_time, preferred_satellite=18):
    query = search_goes_xrs(start_time, end_time, preferred_satellite=preferred_satellite)
    file_count = getattr(query, 'file_num', None)

    if file_count == 0:
        raise RuntimeError(f'No GOES XRS files were found for {start_time} to {end_time}.')

    files = Fido.fetch(query, path=str(cache_dir / '{file}'))
    ts = TimeSeries(list(files), concatenate=True)
    df = ts.to_dataframe()

    if df.empty:
        raise RuntimeError('The downloaded GOES TimeSeries was empty.')

    df.index = pd.to_datetime(df.index)
    df = df.sort_index()
    df = df[~df.index.duplicated(keep='first')]
    df, short_col, long_col = pick_flux_columns(df)
    df = df[[short_col, long_col]].rename(columns={short_col: 'xrsa', long_col: 'xrsb'})
    df = df.replace([np.inf, -np.inf], np.nan).dropna()
    df = df[(df['xrsa'] > 0) & (df['xrsb'] > 0)]
    return df


def smooth_flux(series, window_points=7):
    return series.rolling(window=window_points, center=True, min_periods=1).median()


def classify_goes_flux(flux_value):
    if flux_value >= 1e-4:
        return f"X{flux_value / 1e-4:.1f}"
    if flux_value >= 1e-5:
        return f"M{flux_value / 1e-5:.1f}"
    if flux_value >= 1e-6:
        return f"C{flux_value / 1e-6:.1f}"
    if flux_value >= 1e-7:
        return f"B{flux_value / 1e-7:.1f}"
    if flux_value >= 1e-8:
        return f"A{flux_value / 1e-8:.1f}"
    return f"Below A ({flux_value:.2e})"


def measure_flare_properties(df, label, start_time, end_time, smoothing_points=7):
    working = df.copy()
    working['xrsb_smooth'] = smooth_flux(working['xrsb'], window_points=smoothing_points)
    working['xrsa_smooth'] = smooth_flux(working['xrsa'], window_points=smoothing_points)

    peak_time = working['xrsb_smooth'].idxmax()
    peak_flux = float(working['xrsb_smooth'].max())
    baseline_count = max(5, min(len(working) // 8, 40))
    baseline_flux = float(working['xrsb_smooth'].iloc[:baseline_count].median())
    peak_to_baseline = peak_flux / max(baseline_flux, 1e-12)

    event_detected = bool((peak_flux >= 1e-6) and (peak_to_baseline >= 1.5))

    if event_detected:
        trigger_flux = baseline_flux + 0.05 * (peak_flux - baseline_flux)
        pre_peak = working.loc[:peak_time, 'xrsb_smooth']
        post_peak = working.loc[peak_time:, 'xrsb_smooth']
        start_candidates = pre_peak[pre_peak >= trigger_flux]
        end_candidates = post_peak[post_peak >= trigger_flux]

        start_time_est = start_candidates.index[0]
        end_time_est = end_candidates.index[-1]
        duration_minutes = (end_time_est - start_time_est).total_seconds() / 60.0
        rise_minutes = (peak_time - start_time_est).total_seconds() / 60.0
        decay_minutes = (end_time_est - peak_time).total_seconds() / 60.0
    else:
        start_time_est = pd.NaT
        end_time_est = pd.NaT
        duration_minutes = np.nan
        rise_minutes = np.nan
        decay_minutes = np.nan

    peak_short_flux = float(working.loc[peak_time, 'xrsa_smooth'])
    short_to_long_ratio = peak_short_flux / peak_flux

    return {
        'Event window': label,
        'Window start': pd.Timestamp(start_time),
        'Window end': pd.Timestamp(end_time),
        'Peak time (UTC)': peak_time,
        'Peak flux (W/m^2)': peak_flux,
        'Flare class': classify_goes_flux(peak_flux),
        'Baseline flux (W/m^2)': baseline_flux,
        'Peak/baseline ratio': peak_to_baseline,
        'Strong isolated flare?': event_detected,
        'Estimated start (UTC)': start_time_est,
        'Estimated end (UTC)': end_time_est,
        'Estimated duration (min)': duration_minutes,
        'Rise time (min)': rise_minutes,
        'Decay time (min)': decay_minutes,
        'Short/long flux ratio at peak': short_to_long_ratio
    }


print('Helper functions ready. We can now load the selected event windows and compare them consistently.')

In [ ]:
print('Loading GOES X-ray data for each curated window...')

# Suppress the asyncio garbage collection traceback from printing to stderr LIVE COMPATIBLE
logging.getLogger("asyncio").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=UserWarning, module="asyncio")

# EDIT THIS VALUE: smoothing window for the first diagnostic pass
default_smoothing_points = 7

event_data = {}
summary_rows = []

for label, config in flare_windows.items():
    print(f"Loading: {label} | {config['start']} to {config['end']} UTC")
    df = load_goes_window(config['start'], config['end'], preferred_satellite=18)
    event_data[label] = df
    summary = measure_flare_properties(
        df,
        label=label,
        start_time=config['start'],
        end_time=config['end'],
        smoothing_points=default_smoothing_points
    )
    summary_rows.append(summary)

event_summary = pd.DataFrame(summary_rows).sort_values('Peak flux (W/m^2)', ascending=False).reset_index(drop=True)

summary_view = event_summary[[
    'Event window',
    'Window start',
    'Window end',
    'Peak time (UTC)',
    'Peak flux (W/m^2)',
    'Flare class',
    'Estimated duration (min)',
    'Peak/baseline ratio',
    'Strong isolated flare?'
]].copy()

summary_view['Window start'] = summary_view['Window start'].dt.strftime('%Y-%m-%d %H:%M')
summary_view['Window end'] = summary_view['Window end'].dt.strftime('%Y-%m-%d %H:%M')
summary_view['Peak time (UTC)'] = summary_view['Peak time (UTC)'].dt.strftime('%Y-%m-%d %H:%M:%S')
summary_view['Peak flux (W/m^2)'] = summary_view['Peak flux (W/m^2)'].map(lambda value: f'{value:.2e}')
summary_view['Estimated duration (min)'] = summary_view['Estimated duration (min)'].map(lambda value: 'Not isolated' if pd.isna(value) else f'{value:.1f}')
summary_view['Peak/baseline ratio'] = summary_view['Peak/baseline ratio'].map(lambda value: f'{value:.1f}')

display(summary_view)
print('Flare data loaded and summarised. We now have a reproducible event table to interpret.')

## 4. Inspecting One Real Flare in Detail

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Goal:</b> Focus on the strongest window in our curated set and inspect how the X-ray flux rises, peaks, and decays.
</div>

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Interpretation idea:</b> The long channel is the usual reference for GOES flare classification. The short channel can still be useful because it helps us see that the flare is brightening across more than one X-ray band.
</div>

Run the next cell and inspect the annotations. Notice the log-scale y-axis: flare classes differ by powers of ten.

In [ ]:
print('Preparing a detailed view of the strongest flare window...')

strongest_event_label = event_summary.iloc[0]['Event window']
strongest_info = event_summary.set_index('Event window').loc[strongest_event_label]
strongest_df = event_data[strongest_event_label].copy()
strongest_df['xrsa_smooth'] = smooth_flux(strongest_df['xrsa'], window_points=default_smoothing_points)
strongest_df['xrsb_smooth'] = smooth_flux(strongest_df['xrsb'], window_points=default_smoothing_points)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=strongest_df.index,
    y=strongest_df['xrsb'],
    mode='lines',
    line=dict(color='rgba(76, 114, 176, 0.35)', width=1),
    name='Long channel raw'
))
fig.add_trace(go.Scatter(
    x=strongest_df.index,
    y=strongest_df['xrsb_smooth'],
    mode='lines',
    line=dict(color='#4C72B0', width=3),
    name='Long channel smoothed'
))
fig.add_trace(go.Scatter(
    x=strongest_df.index,
    y=strongest_df['xrsa_smooth'],
    mode='lines',
    line=dict(color='#DD8452', width=2),
    name='Short channel smoothed'
))

for level, label, color in [
    (1e-6, 'C threshold', 'gray'),
    (1e-5, 'M threshold', '#C44E52'),
    (1e-4, 'X threshold', '#8172B2')
]:
    fig.add_hline(y=level, line_dash='dash', line_color=color, opacity=0.6)
    fig.add_annotation(
        x=strongest_df.index[int(len(strongest_df) * 0.93)],
        y=level,
        text=label,
        showarrow=False,
        font=dict(size=11, color=color),
        bgcolor='rgba(255,255,255,0.7)'
    )

peak_time = strongest_info['Peak time (UTC)']
peak_flux = strongest_info['Peak flux (W/m^2)']

fig.add_vline(x=peak_time, line_color='black', line_dash='dot')
fig.add_annotation(
    x=peak_time,
    y=peak_flux,
    text=f"Peak: {strongest_info['Flare class']}<br>{pd.Timestamp(peak_time).strftime('%Y-%m-%d %H:%M:%S UTC')}",
    showarrow=True,
    arrowhead=2,
    yshift=15,
    bgcolor='rgba(255,255,255,0.8)'
)

if pd.notna(strongest_info['Estimated start (UTC)']):
    fig.add_vline(x=strongest_info['Estimated start (UTC)'], line_color='green', line_dash='dash')
    fig.add_vline(x=strongest_info['Estimated end (UTC)'], line_color='red', line_dash='dash')

fig.update_layout(
    template='plotly_white',
    height=520,
    width=1000,
    title=f'Detailed GOES view: {strongest_event_label}',
    xaxis_title='Time (UTC)',
    yaxis_title='X-ray flux (W/m^2)',
    yaxis_type='log',
    legend_title='Measurement'
)

fig.show()
print(f"Detailed event view ready. The strongest window in our set peaks at {strongest_info['Flare class']}.")

## 5. Comparing Multiple Windows

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Goal:</b> Compare quiet and active solar windows, rank them by peak intensity, and look at how duration and flare class change from one event to the next.
</div>

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Scientific idea:</b> Peak flux matters, but so does flare shape. Two events can have different rise and decay behaviour even if they both look dramatic at first glance.
</div>

In [ ]:
print('Building a comparison view across all curated event windows...')

ordered_labels = event_summary['Event window'].tolist()
comparison_fig = make_subplots(
    rows=len(ordered_labels),
    cols=1,
    shared_yaxes=True,
    shared_xaxes=False,
    vertical_spacing=0.04,
    subplot_titles=ordered_labels
)

for row_number, label in enumerate(ordered_labels, start=1):
    df = event_data[label].copy()
    df['xrsb_smooth'] = smooth_flux(df['xrsb'], window_points=default_smoothing_points)
    event_info = event_summary.set_index('Event window').loc[label]

    comparison_fig.add_trace(
        go.Scatter(
            x=df.index,
            y=df['xrsb_smooth'],
            mode='lines',
            line=dict(width=2),
            name=label,
            showlegend=False
        ),
        row=row_number,
        col=1
    )

    comparison_fig.add_vline(
        x=event_info['Peak time (UTC)'],
        line_dash='dot',
        line_color='black',
        row=row_number,
        col=1
    )

    comparison_fig.add_annotation(
        x=event_info['Peak time (UTC)'],
        y=event_info['Peak flux (W/m^2)'],
        text=event_info['Flare class'],
        showarrow=True,
        arrowhead=2,
        bgcolor='rgba(255,255,255,0.75)',
        row=row_number,
        col=1
    )

comparison_fig.update_yaxes(type='log', title_text='Flux', row=row_number, col=1)

comparison_fig.update_layout(
    template='plotly_white',
    height=900,
    width=1000,
    title='GOES long-channel X-ray flux across curated solar-activity windows'
)
comparison_fig.update_xaxes(title_text='Time (UTC)', row=len(ordered_labels), col=1)
comparison_fig.show()

print('Comparison plot ready. The quieter baseline and the stronger flare windows now sit side by side.')

In [ ]:
print('Summarising flare intensity and duration with a polished dashboard...')

dashboard_data = event_summary.copy()
dashboard_data['Duration for plot (min)'] = dashboard_data['Estimated duration (min)'].fillna(0)

dashboard_fig = make_subplots(specs=[[{'secondary_y': True}]])

dashboard_fig.add_trace(
    go.Bar(
        x=dashboard_data['Event window'],
        y=dashboard_data['Peak flux (W/m^2)'],
        name='Peak flux',
        marker_color='#4C72B0',
        hovertemplate='%{x}<br>Peak flux: %{y:.2e} W/m^2<extra></extra>'
    ),
    secondary_y=False
)

dashboard_fig.add_trace(
    go.Scatter(
        x=dashboard_data['Event window'],
        y=dashboard_data['Duration for plot (min)'],
        mode='lines+markers',
        name='Estimated duration',
        line=dict(color='#DD8452', width=3),
        marker=dict(size=10),
        hovertemplate='%{x}<br>Estimated duration: %{y:.1f} min<extra></extra>'
    ),
    secondary_y=True
)

dashboard_fig.update_layout(
    template='plotly_white',
    height=520,
    width=1000,
    title='Event ranking: intensity versus estimated duration'
)
dashboard_fig.update_yaxes(title_text='Peak flux (W/m^2)', type='log', secondary_y=False)
dashboard_fig.update_yaxes(title_text='Estimated duration (min)', secondary_y=True)
dashboard_fig.update_xaxes(title_text='Curated event window')
dashboard_fig.show()

print('Summary dashboard ready. This makes it easier to compare dramatic-looking events quantitatively.')

<div style="border:1px solid #ffeeba; background:#fff3cd; color:#856404; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Exercise:</b> Go back to the data-loading cell and change the smoothing window from <b>7</b> to <b>5</b> or <b>11</b>. Then rerun the detailed and comparison cells. Do the peak time and estimated duration stay nearly the same, or do they shift noticeably?
</div>

## 6. Interactive Flare Explorer

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Interactivity:</b> This is where the notebook becomes exploratory. You can pick a flare window, adjust the smoothing, and decide whether to show the short channel. That lets you see how different display choices affect interpretation.
</div>

<div style="border:1px solid #ffeeba; background:#fff3cd; color:#856404; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Try this:</b> Start with the quiet window, then switch to the strongest one. After that, increase the smoothing and look for changes in peak sharpness and duration estimates.
</div>

Run the next cell, then use the controls.

In [ ]:
print('Building the interactive flare explorer...')

def explore_flare(event_label, smoothing_points=7, show_short_channel=True):
    df = event_data[event_label].copy()
    summary = measure_flare_properties(
        df,
        label=event_label,
        start_time=flare_windows[event_label]['start'],
        end_time=flare_windows[event_label]['end'],
        smoothing_points=smoothing_points
    )

    df['xrsb_smooth'] = smooth_flux(df['xrsb'], window_points=smoothing_points)
    df['xrsa_smooth'] = smooth_flux(df['xrsa'], window_points=smoothing_points)

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=df.index,
        y=df['xrsb'],
        mode='lines',
        line=dict(color='rgba(76, 114, 176, 0.25)', width=1),
        name='Long channel raw'
    ))
    fig.add_trace(go.Scatter(
        x=df.index,
        y=df['xrsb_smooth'],
        mode='lines',
        line=dict(color='#4C72B0', width=3),
        name='Long channel smoothed'
    ))

    if show_short_channel:
        fig.add_trace(go.Scatter(
            x=df.index,
            y=df['xrsa_smooth'],
            mode='lines',
            line=dict(color='#DD8452', width=2),
            name='Short channel smoothed'
        ))

    for level, label, color in [
        (1e-6, 'C', 'gray'),
        (1e-5, 'M', '#C44E52'),
        (1e-4, 'X', '#8172B2')
    ]:
        fig.add_hline(y=level, line_dash='dash', line_color=color, opacity=0.5)

    fig.add_vline(x=summary['Peak time (UTC)'], line_color='black', line_dash='dot')

    if pd.notna(summary['Estimated start (UTC)']):
        fig.add_vline(x=summary['Estimated start (UTC)'], line_color='green', line_dash='dash')
        fig.add_vline(x=summary['Estimated end (UTC)'], line_color='red', line_dash='dash')

    fig.update_layout(
        template='plotly_white',
        height=500,
        width=980,
        title=f'Interactive GOES flare explorer: {event_label}',
        xaxis_title='Time (UTC)',
        yaxis_title='X-ray flux (W/m^2)',
        yaxis_type='log'
    )
    fig.show()

    info_df = pd.DataFrame({
        'Property': [
            'Peak time (UTC)',
            'Peak flux (W/m^2)',
            'Flare class',
            'Estimated duration (min)',
            'Rise time (min)',
            'Decay time (min)',
            'Peak/baseline ratio',
            'Strong isolated flare?'
        ],
        'Value': [
            pd.Timestamp(summary['Peak time (UTC)']).strftime('%Y-%m-%d %H:%M:%S'),
            f"{summary['Peak flux (W/m^2)']:.2e}",
            summary['Flare class'],
            'Not isolated' if pd.isna(summary['Estimated duration (min)']) else f"{summary['Estimated duration (min)']:.1f}",
            'Not isolated' if pd.isna(summary['Rise time (min)']) else f"{summary['Rise time (min)']:.1f}",
            'Not isolated' if pd.isna(summary['Decay time (min)']) else f"{summary['Decay time (min)']:.1f}",
            f"{summary['Peak/baseline ratio']:.1f}",
            str(summary['Strong isolated flare?'])
        ]
    })
    display(info_df)


widgets.interact(
    explore_flare,
    event_label=widgets.Dropdown(
        options=list(flare_windows.keys()),
        value=strongest_event_label,
        description='Event:'
    ),
    smoothing_points=widgets.IntSlider(
        value=7,
        min=1,
        max=21,
        step=2,
        description='Smoothing:',
        continuous_update=False
    ),
    show_short_channel=widgets.Checkbox(
        value=True,
        description='Show short channel'
    )
)

print('Interactive explorer ready. Try one control at a time so the effect is easy to interpret.')

## 7. Responsible Interpretation and Workflow Limits

<div style="border:1px solid #f5c6cb; background:#f8d7da; color:#721c24; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Important caution:</b> This notebook is a strong teaching workflow, but it is not a full operational space-weather pipeline. A flare time series tells us a lot, but it does not tell us everything.
</div>

Some important limitations to keep in mind:

- <b>Full-disk measurement:</b> GOES XRS sees the integrated X-ray output of the whole Sun, not a spatially resolved image of one active region.
- <b>Classification is not the whole story:</b> a flare's X-ray class is useful, but it does not automatically tell us everything about particle events, coronal mass ejections, or impacts at Earth.
- <b>Our duration estimate is heuristic:</b> we used a transparent threshold-based method for teaching, not an official event catalogue algorithm.
- <b>Smoothing can matter:</b> changing the smoothing window can slightly shift the measured peak time or duration, especially in noisier or more complex windows.
- <b>Context is limited:</b> without imaging, spectroscopy, magnetic data, or coronagraph observations, we should avoid over-interpreting physical causes.
- <b>Data retrieval can vary:</b> archived remote services and instrument coverage are excellent, but not every query returns exactly the same structure forever.

That is a normal part of scientific work. Responsible interpretation means being clear about what the data measure, what the workflow estimates, and what still requires extra evidence.

## Take-away

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    Congratulations! You have used real GOES X-ray data to inspect solar activity as a time-domain astronomy problem. You selected event windows, loaded data reproducibly, identified flare peaks, estimated simple event properties, compared quiet and active intervals, and explored the results interactively.
</div>

Strong next steps could include:
- comparing these flare windows with official NOAA event catalogues
- adding EUV or magnetogram context for the same solar period
- comparing short-channel and long-channel evolution in more detail
- investigating whether the largest X-ray peak also had the longest decay
- expanding the workflow into a broader space-weather event notebook

This is a great example of how modern astronomy and heliophysics turn a time series into a story about a dynamic star.

![Noteable license](https://raw.githubusercontent.com/YusufNik/Noteable/refs/heads/main/images/Noteable%20Notebook%20Footer.png)